<a href="https://colab.research.google.com/github/mbithi002/generativeai/blob/main/mbithi002_new_implementation_research_logs_on_Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project - Text generation and Tokenization.

Load a simple text generation model, generate text and tokenize the text then inspect the input ids

In [ ]:
import torch
from transformers import pipeline

Model https://huggingface.co/google/gemma-2-2b-it , to access the model go ahead and navigate to the link and accept the terms and conditions to be added to the access list.

After being added to the access list, create a Huggingface api key on their website with the relevant permissions and add a new colab secret with name HF_TOKEN and your generated key as the value.

In [ ]:
from google.colab import userdata
pipe = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",
    token=userdata.get("HF_TOKEN"),
)

In [ ]:
messages = [
    {"role": "user", "content": "Write a poem about the Genghis Khan's reign. Not less than 10 sentences."},
]

In [ ]:
outputs = pipe(messages, max_new_tokens=512)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

Let's now tokenize the above generated response.

In [ ]:
from transformers import AutoTokenizer

It is important to note that every model has it's own tokenizer. So here we opt to load gemma's tokenizer.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# use assistant_response above as Sample text
# Perform tokenization
tokens = tokenizer.tokenize(assistant_response)
input_ids = tokenizer.encode(assistant_response)
input_ids_with_special = tokenizer.encode(assistant_response, add_special_tokens=True)

# Decode back to text
decoded_text = tokenizer.decode(input_ids_with_special)

In [ ]:
# Print results
print("="*50)
print("TOKENIZATION RESULTS")
print("="*50)
print(f"Original text: '{assistant_response}'")
print(f"\nTokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"\nInput IDs (with special tokens): {input_ids_with_special}")
print(f"Length of input IDs: {len(input_ids_with_special)}")

In [ ]:
print("\n" + "="*50)
print("TOKEN TO ID MAPPING")
print("="*50)
for i, token in enumerate(tokens):
    print(f"Token: '{token}' -> ID: {tokenizer.convert_tokens_to_ids(token)}")

In [ ]:
# Show special tokens
print("\n" + "="*50)
print("SPECIAL TOKENS")
print("="*50)
print(f"[CLS] token ID: {tokenizer.cls_token_id}")
print(f"[SEP] token ID: {tokenizer.sep_token_id}")
print(f"[PAD] token ID: {tokenizer.pad_token_id}")
print(f"[MASK] token ID: {tokenizer.mask_token_id}")


For attention based tokenizers, in the Transformers architecture see below. Read Research Paper 'Attention is All you need' by GOogle

In [ ]:
# Create attention mask
attention_mask = [1] * len(input_ids_with_special)

print("\n" + "="*50)
print("ATTENTION MASK")
print("="*50)
print(f"Input IDs: {input_ids_with_special}")
print(f"Attention mask: {attention_mask}")
print("(1 means attend to token, 0 means ignore/padding)")


In [ ]:
# Demonstrate with batch processing and padding
print("\n" + "="*50)
print("BATCH PROCESSING WITH PADDING")
print("="*50)

# Two texts of different lengths
texts = [
    "I love AI",
    "I love artificial intelligence and machine learning!"
]

# Tokenize with padding and truncation
batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=10,
    return_tensors="pt"
)

print(f"Texts: {texts}")
print(f"\nInput IDs (padded):\n{batch['input_ids']}")
print(f"\nAttention masks (1=real token, 0=padding):\n{batch['attention_mask']}")


In [ ]:
# Visual explanation
print("\n" + "="*50)
print("VISUAL EXPLANATION")
print("="*50)
print("Original: 'I love artificial intelligence and machine learning!'")
print("Tokens:   ['i', 'love', 'artificial', 'intelligence', 'and', 'machine', 'learning', '!']")
print("IDs:      [101, 1045, 2293, 4605, 3293, 1998, 3052, 2750, 999, 102]")
print("          ^^^^                                   ^^^^")
print("         [CLS]                                  [SEP]")